# 🎯 EduTutor — Notebook 2: Unsloth Fine-Tuning Pipeline

**Project:** EduTutor-Unsloth — A Neurodiversity-Affirming AI Tutor  
**Competition:** [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)  
**Tracks:** Future of Education + Unsloth Tech Track  
**Model:** Gemma 4 E4B (4B params, 128K context, multimodal, Apache 2.0)  

---

## Pipeline

```
Base Gemma 4 E4B
       │
       ▼
┌─────────────────┐
│  Stage 1: SFT   │  QLoRA + Unsloth → Teach the pedagogical persona
│  (Supervised)    │  Dataset: sft_train.jsonl from Notebook 1
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Stage 2: DPO   │  Direct Preference Optimization → Align away from
│  (Alignment)    │  answer-giving toward productive struggle
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Stage 3: Export │  Merge adapters → Export GGUF for local deployment
└─────────────────┘
```

### Hardware Requirements
- **GPU:** T4 (16GB) on Kaggle/Colab — sufficient for E4B QLoRA (~17GB peak with gradient checkpointing)
- **RAM:** 12GB+
- **Disk:** ~10GB for model weights + adapters

## 1. Environment Setup

In [ ]:
%%capture
# Install Unsloth — always use latest for Gemma 4 compatibility
!pip install -U unsloth
!pip install -q jsonlines datasets trl transformers accelerate bitsandbytes

In [ ]:
import torch
import json
import jsonlines
from pathlib import Path
from datasets import Dataset

# Import the shared system prompt (single source of truth)
from local_model import EDUTUTOR_SYSTEM_PROMPT

# Verify GPU
assert torch.cuda.is_available(), "GPU required — enable T4 in Kaggle settings"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.version.cuda}")

## 2. Load Gemma 4 E4B with Unsloth

Unsloth patches the model for 2x faster training and 60% less VRAM. We load in 4-bit quantization (QLoRA) to fit on a T4.

In [ ]:
from unsloth import FastLanguageModel

# ---------- Model Configuration ----------
MODEL_NAME = "google/gemma-4-e4b"  # Gemma 4 Effective 4B
MAX_SEQ_LENGTH = 4096              # conversation context window
LOAD_IN_4BIT = True                # QLoRA quantization
DTYPE = None                       # auto-detect (float16 for T4)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=DTYPE,
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   Quantization: {'4-bit QLoRA' if LOAD_IN_4BIT else 'Full precision'}")

## 3. Configure LoRA Adapters

We add LoRA adapters to the attention and MLP layers. Configuration choices:
- **r=32:** Higher rank for richer pedagogical personality capture
- **alpha=64:** Standard 2× rank scaling
- **Target modules:** All attention projections + gate/up/down MLPs for maximum expressiveness

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                          # LoRA rank — higher for complex persona
    lora_alpha=64,                 # scaling factor (2× rank)
    lora_dropout=0.05,             # slight regularization
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj",      # MLP
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

# Print trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA adapters attached")
print(f"   Trainable params: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Total params: {total:,}")

## 4. Load and Format SFT Dataset

We load the synthetic data from Notebook 1 and format it using the Gemma 4 chat template.

In [ ]:
DATA_DIR = Path("data")


def load_sft_data(path: Path) -> Dataset:
    """Load SFT JSONL and format for the Gemma 4 chat template."""
    examples = []
    with jsonlines.open(path) as reader:
        for item in reader:
            messages = item["messages"]
            # Apply the Gemma 4 chat template
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            examples.append({"text": text})
    
    return Dataset.from_list(examples)


sft_train = load_sft_data(DATA_DIR / "sft_train.jsonl")
sft_val = load_sft_data(DATA_DIR / "sft_validation.jsonl")

print(f"✅ SFT data loaded")
print(f"   Train: {len(sft_train)} examples")
print(f"   Validation: {len(sft_val)} examples")
print(f"\n📝 Sample (first 500 chars):")
print(sft_train[0]["text"][:500])

## 5. Stage 1: Supervised Fine-Tuning (SFT)

This stage teaches the model the **EduTutor persona**: communication style, scaffolding patterns, emotional co-regulation, and profile-specific strategies.

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    # --- Output ---
    output_dir="./edututor-sft-checkpoints",
    
    # --- Training ---
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,      # effective batch size = 8
    warmup_ratio=0.1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    
    # --- Precision ---
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=50,
    
    # --- Saving ---
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    
    # --- Logging ---
    logging_steps=10,
    report_to="none",
    
    # --- Sequence ---
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,                       # pack short sequences for throughput
    dataset_text_field="text",
    
    # --- Seed ---
    seed=42,
)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_train,
    eval_dataset=sft_val,
    args=sft_config,
)

print("✅ SFT Trainer configured")
print(f"   Epochs: {sft_config.num_train_epochs}")
print(f"   Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"   Learning rate: {sft_config.learning_rate}")

In [ ]:
# ---------- Train SFT ----------
print("🚀 Starting SFT training...")
sft_result = sft_trainer.train()

print(f"\n✅ SFT Training complete!")
print(f"   Total steps: {sft_result.global_step}")
print(f"   Final train loss: {sft_result.training_loss:.4f}")

# Save the SFT adapter
SFT_ADAPTER_DIR = "./edututor-sft-adapter"
model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)
print(f"   Adapter saved to: {SFT_ADAPTER_DIR}")

### 5.1 Quick Sanity Check: SFT Model Response

Before moving to DPO, let's verify the SFT model has learned the basic persona.

In [ ]:
FastLanguageModel.for_inference(model)  # switch to inference mode

# Use the shared system prompt from local_model.py (single source of truth)
test_messages = [
    {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
    {"role": "user", "content": "I HATE fractions! I can't do this and I want to quit! I'm so STUPID!"}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=300,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print("🧒 Student: I HATE fractions! I can't do this and I want to quit! I'm so STUPID!")
print(f"\n🤖 EduTutor (SFT):")
print(response)
print("\n--- Sanity Checks ---")
print(f"  Contains empathy/validation: {'✅' if any(w in response.lower() for w in ['feel', 'hear', 'okay', 'understand', 'hard']) else '❌'}")
print(f"  Avoids giving answer: {'✅' if 'the answer is' not in response.lower() else '❌'}")
print(f"  Short sentences: {'✅' if len(response.split('.')[0].split()) < 20 else '⚠️  First sentence may be too long'}")

## 6. Stage 2: DPO Alignment

DPO further refines the model by training it to **prefer** scaffolding responses over answer-giving responses. This is the key alignment step that prevents the model from reverting to generic "helpful assistant" behavior.

In [ ]:
FastLanguageModel.for_training(model)  # switch back to training mode

# Ensure pad token is set for proper DPO collation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✅ Set pad_token = eos_token ({tokenizer.eos_token})")


def load_dpo_data(path: Path) -> Dataset:
    """Load DPO JSONL into HF Dataset format expected by TRL's DPOTrainer."""
    examples = []
    with jsonlines.open(path) as reader:
        for item in reader:
            # Format prompt messages
            prompt_text = tokenizer.apply_chat_template(
                item["prompt"],
                tokenize=False,
                add_generation_prompt=True,
            )
            examples.append({
                "prompt": prompt_text,
                "chosen": item["chosen"][0]["content"],
                "rejected": item["rejected"][0]["content"],
            })
    
    return Dataset.from_list(examples)


dpo_train = load_dpo_data(DATA_DIR / "dpo_train.jsonl")
dpo_val = load_dpo_data(DATA_DIR / "dpo_validation.jsonl")

print(f"✅ DPO data loaded")
print(f"   Train: {len(dpo_train)} preference pairs")
print(f"   Validation: {len(dpo_val)} preference pairs")

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir="./edututor-dpo-checkpoints",
    
    # --- Training ---
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,       # effective batch size = 8
    learning_rate=5e-5,                  # lower LR for alignment
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    
    # --- DPO Specific ---
    beta=0.1,                            # KL penalty coefficient
    loss_type="sigmoid",                 # standard DPO loss
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=1024,
    
    # --- Precision ---
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=25,
    
    # --- Saving ---
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    
    # --- Logging ---
    logging_steps=5,
    report_to="none",
    
    seed=42,
)

dpo_trainer = DPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dpo_train,
    eval_dataset=dpo_val,
    args=dpo_config,
)

print("✅ DPO Trainer configured")
print(f"   Beta (KL penalty): {dpo_config.beta}")
print(f"   Learning rate: {dpo_config.learning_rate}")

In [ ]:
# ---------- Train DPO ----------
print("🚀 Starting DPO alignment...")
dpo_result = dpo_trainer.train()

print(f"\n✅ DPO Alignment complete!")
print(f"   Total steps: {dpo_result.global_step}")
print(f"   Final train loss: {dpo_result.training_loss:.4f}")

# Save the DPO-aligned adapter
DPO_ADAPTER_DIR = "./edututor-dpo-adapter"
model.save_pretrained(DPO_ADAPTER_DIR)
tokenizer.save_pretrained(DPO_ADAPTER_DIR)
print(f"   Adapter saved to: {DPO_ADAPTER_DIR}")

## 7. Stage 3: Export

We merge the LoRA adapters into the base model and export to multiple formats:
- **GGUF (Q4_K_M):** For llama.cpp / Ollama local deployment
- **Safetensors:** For HuggingFace Hub / vLLM serving

In [ ]:
# ---------- Merge and Export ----------
MERGED_DIR = "./edututor-merged"
GGUF_DIR = "./edututor-gguf"

# Export to GGUF for local deployment (Ollama/llama.cpp)
print("📦 Exporting to GGUF (Q4_K_M quantization)...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",  # good quality/size balance
)
print(f"   ✅ GGUF saved to: {GGUF_DIR}")

# Also save merged safetensors for HF Hub
print("\n📦 Exporting merged safetensors...")
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print(f"   ✅ Merged model saved to: {MERGED_DIR}")

# List exported files
import os
print(f"\n📁 GGUF files:")
for f in os.listdir(GGUF_DIR):
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1e6
    print(f"   {f} ({size_mb:.1f} MB)")

## 8. Final Verification: Post-DPO Response Quality

Compare the DPO-aligned model against the same test prompt to verify alignment worked.

In [ ]:
FastLanguageModel.for_inference(model)

test_scenarios = [
    {
        "label": "RED Zone — Crisis (Dyscalculia)",
        "messages": [
            {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": "I CAN'T DO MATH! I'm the WORST in my class! Everyone finishes before me and I NEVER get it right! I want to QUIT FOREVER!"},
        ]
    },
    {
        "label": "GREEN Zone — Ready to Learn (Dyslexia)",
        "messages": [
            {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": "Okay, I'm ready to try reading. Can we practice the 'sh' sound today?"},
        ]
    },
    {
        "label": "YELLOW Zone — Frustrated (ADHD)",
        "messages": [
            {"role": "system", "content": EDUTUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": "Ugh, this is taking forever. Can we just skip fractions? I already know them. Well, maybe not. I don't know. This is boring."},
        ]
    },
]

for scenario in test_scenarios:
    inputs = tokenizer.apply_chat_template(
        scenario["messages"],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    
    print(f"\n{'━' * 60}")
    print(f"📋 Scenario: {scenario['label']}")
    print(f"{'━' * 60}")
    print(f"🧒 Student: {scenario['messages'][-1]['content']}")
    print(f"\n🤖 EduTutor (SFT+DPO):")
    print(response)
    print()

## Summary

| Stage | What It Did | Output |
|---|---|---|
| **SFT** | Taught the model the EduTutor persona, scaffolding patterns, and profile-specific strategies | LoRA adapter in `edututor-sft-adapter/` |
| **DPO** | Aligned the model to prefer productive struggle over answer-giving | LoRA adapter in `edututor-dpo-adapter/` |
| **Export** | Merged adapters and exported for deployment | GGUF in `edututor-gguf/`, safetensors in `edututor-merged/` |

### Next Step

→ **Notebook 3: `03_evaluation_harness.ipynb`** — Formally evaluate the fine-tuned model against base Gemma 4 using custom pedagogical rubrics and LLM-as-Judge.